In [ ]:
# env 로드
from dotenv import load_dotenv
load_dotenv()

In [ ]:
# 필요한거 import 
import torch
from transformers import BitsAndBytesConfig
from transformers import AutoTokenizer, AutoModelForCausalLM

In [ ]:
# 4비트 양자화
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

In [ ]:
# 모델 로드
model_id = 'Bllossom/llama-3.2-Korean-Bllossom-3B'

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    quantization_config=bnb_config 
)

In [ ]:
# 키워드 넣으면 생성해주는 함수로 만들기
def create_trpg_world(keywords):
    instruction = f"{keywords} 이 키워드들을 사용해서 trpg에 사용할 세계관을 짜줘."

    messages = [
        {"role": "system", "content": "당신은 한국인 TRPG 게임 마스터입니다. 한국어 위주로만 사용하고, 듣기만 해도 배경이 그려지는 듯하게 설명해주세요."},
        {"role": "user", "content": f"{instruction}"}
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True  # 딕셔너리 형태로 반환 강제
    ).to(model.device)

    terminators = [
        tokenizer.convert_tokens_to_ids("<|end_of_text|>"),
        tokenizer.convert_tokens_to_ids("<|eot_id|>")
    ]

    outputs = model.generate(
        **inputs,
        max_new_tokens=512, 
        eos_token_id=terminators,
        do_sample=True,
        temperature=0.55,
        top_p=0.8,
        repetition_penalty=1.2 # 같은 말 반복 방지
    )

    return tokenizer.decode(outputs[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True)


In [ ]:
# 요약(정보은닉) 및 다음 행동 관련 질문
def summarize_for_player(master_lore):
    system_prompt = (
        "당신은 TRPG 마스터입니다. 제시된 상세 설정을 바탕으로 플레이어에게 직접 말해줄 도입부 대사를 작성하세요. "
        "1. 반드시 한국어로만 말하세요. 2. 3문장 이내로 요약하세요. 3. 마지막은 항상 다음 행동을 묻는 질문으로 끝내세요."
    )
    
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"상세 설정: {master_lore}"}
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    ).to(model.device)

    terminators = [
        tokenizer.convert_tokens_to_ids("<|end_of_text|>"),
        tokenizer.convert_tokens_to_ids("<|eot_id|>")
    ]

    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        eos_token_id=terminators,
        do_sample=True,
        temperature=0.5, # 요약의 정확도를 위해 온도를 낮춤
        top_p=0.9
    )

    return tokenizer.decode(outputs[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True)

In [ ]:
print("✅ TRPG 엔진 준비 완료!")